# Day 4 — Decision layer: from a ranked list to a decision

Day 3 chose a champion — the **response baseline**, a statistical tie broken on
profit and parsimony — and this session registered it as a real artifact
(`pipelines/train.py` → `models/champion.joblib` + provenance sidecar). This
notebook turns that ranking into *decisions*:

1. **provenance** — what the registered model actually is;
2. **stratified sampling, made visible** — a balance report, plus the
   money-outcome stratification option;
3. **profit curve** — how deep to target, and where break-even sits;
4. **scenario comparison** — the champion's money vs random / target-all /
   do-nothing at the same budget;
5. **multi-treatment** — "which email, if any?", scored honestly with IPW;
6. **policy tree** — readable segment rules, re-scored on the holdout;
7. **fairness audit** — who the policy touches, by `zip_code`;
8. **power analysis** — how big the next experiment must be;
9. the **five typed tools** from `packages/decision/`, end to end.

Every number below comes from the same functions Day 6's API will expose, and
is mirrored into `docs/concepts.md`. Profit is valued on the **money outcome**
(conversions), never the ranking proxy — the Day-3 rule.

## 0. Setup

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")  # causalml/lightgbm/econml emit benign warnings

from packages.causal.benchmark import BusinessAssumptions
from packages.causal.features import (
    build_features,
    encode_splits,
    make_splits,
    split_balance_report,
)
from packages.causal.learners import make_learner
from packages.causal.registry import load_model
from packages.datasets.hillstrom import FEATURES, fetch_hillstrom
from packages.decision import (
    compare_scenarios,
    design_ab_test,
    get_model_card,
    get_segment_profile,
    optimize_budget,
    policy_value_ipw,
)
from packages.plotting import save_fig

pd.set_option("display.width", 140)
RANDOM_STATE = 42
ASSUMPTIONS = BusinessAssumptions.from_yaml()
print("repo root:", ROOT)
print("assumptions:", ASSUMPTIONS)

## 1. The registered champion: provenance, not vibes

The registry refuses to serve a model without its metadata sidecar — data
checksum, split protocol, seed, package versions, and the holdout metrics it
posted under the decided assumptions. The champion is the response baseline
(a LightGBM classifier on the *treated* arm, ranking by purchasability — the
bar every uplift model had to clear and, over 10 seeds, none did).

In [ ]:
champion, meta = load_model("champion")
print("name:      champion")
print(f"learner:   {meta['learner']}   target: {meta['target']}   seed: {meta['random_state']}")
print(f"fit scope: {meta['fit_scope']}")
print(f"split:     {meta['split']}")
print(f"data sha:  {meta['data_sha256'][:16]}...")
print(f"trained:   {meta['created_at']}  @ {meta['git_commit']}")
print("packages: ", {k: v for k, v in meta["packages"].items() if v})
print()
print("registered holdout metrics:")
for k_, v_ in meta["metrics"].items():
    print(f"  {k_:34s} {v_}")

# The data and the champion's own split — identical rows to the registration,
# because the same function + seed produced them.
df = fetch_hillstrom(arm="mens")
X, y, t = build_features(df, target="visit")
splits = make_splits(X, y, t, random_state=RANDOM_STATE)
enc, feature_names = encode_splits(splits)
X_tr, y_tr, t_tr = enc["train"]
X_te, y_te, t_te = enc["test"]
y_money_te = df.loc[X_te.index, "conversion"].to_numpy(dtype=float)
score_champion = np.asarray(champion.predict(X_te.to_numpy())).ravel()
print(f"\ntest rows: {len(X_te)}   converters: {int(y_money_te.sum())}")

**Reading.** The registered Qini (0.0199) and uplift@30% (0.0899) reproduce
notebook 04's single-split table exactly — the artifact traces to the ledger.
Its one-shot profit ($857.60) is *seed 42's* point; the honest envelope is the
10-seed benchmark: **$569 ± 239** at 30% targeted ($0.223/contact vs $0.137
uniform). Nothing here re-trains: `load_model` serves; `pipelines/train.py`
produces; the API on Day 6 will do only the former.

## 2. Stratified sampling, made visible

`make_splits` stratifies on the joint `treatment × outcome` label — a *claim*
that arms and events stay balanced across folds. Section 2 of notebook 04
showed the mechanics; here we make the claim **checkable** with
`split_balance_report`, and demonstrate the new `stratify_extra` option: real
campaigns are paid on the money outcome, which is far rarer than the ranking
target (1% converters vs 15% visitors), so folds can also be balanced on
`conversion` itself.

In [ ]:
splits_extra = make_splits(X, y, t, random_state=RANDOM_STATE, stratify_extra=df["conversion"])

rep_default = split_balance_report(splits)
rep_extra = split_balance_report(splits_extra)

print("DEFAULT protocol (treatment x visit) — the champion's split:")
print(rep_default[["n", "treated_share", "events_treated", "events_control", "smd"]].to_string())
print("\nWITH stratify_extra=conversion (treatment x visit x conversion):")
print(rep_extra[["n", "treated_share", "events_treated", "events_control", "smd"]].to_string())

# Converter share of each arm, fold by fold — the money-outcome balance.
for label, rep in (("default", rep_default), ("extra", rep_extra)):
    tot_t = rep["events_treated"].sum()
    tot_c = rep["events_control"].sum()
    shares_t = (rep["events_treated"] / tot_t).round(4).to_dict()
    shares_c = (rep["events_control"] / tot_c).round(4).to_dict()
    print(f"\n{label:8s} treated-converter share per fold: {shares_t}")
    print(f"{label:8s} control-converter share per fold: {shares_c}")

**Reading.** On Hillstrom the two protocols produce *identical* converter
splits — and that is the finding, not a bug: conversion is nested inside visit
(no converter-without-visitor cells exist in the data), so stratifying on
`treatment x visit` already cuts converters in exact proportion to fold size
(0.64/0.16/0.20 per arm, verified above), and the SMD holds at 0.219 across
all three folds. The report turns that from an assumption into a printed
fact. `stratify_extra` matters on datasets where the money outcome is *not*
nested in the ranking target (e.g. ranking on open-rate while getting paid on
spend); here it is a no-op by construction — pinned by the same tests that
pin the default protocol.

## 3. The profit curve: how deep does the money go?

Sweep the targeting depth on the untouched test split and price each cut with
the decided formula — `value × incremental conversions − cost × contacts`,
valued on **conversions**. The interval at each depth is the treated-control
difference within the targeted set, scaled up; it is the honest width of what
one split can tell us.

In [ ]:
curve = optimize_budget(
    y_money_te, score_champion, t_te, assumptions=ASSUMPTIONS, k_max=0.5, n_depths=25
)

depths = [p.depth for p in curve.points]
profits = [p.profit_usd for p in curve.points]
los = [p.profit_ci_low_usd for p in curve.points]
his = [p.profit_ci_high_usd for p in curve.points]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(depths, los, his, alpha=0.25, color="C0", label="95% CI")
ax.plot(depths, profits, color="C0", lw=2, label="profit (point)")
ax.axhline(0, color="black", lw=0.8)
if curve.break_even_depth is not None:
    ax.axvline(curve.break_even_depth, color="crimson", ls="--", lw=1.2,
               label=f"break-even @ {curve.break_even_depth:.0%}")
if curve.recommended_depth is not None:
    ax.axvline(curve.recommended_depth, color="green", ls="--", lw=1.2,
               label=f"max profit @ {curve.recommended_depth:.0%}")
ax.set_xlabel("targeting depth (share of list)")
ax.set_ylabel("profit on holdout (USD)")
ax.set_title("Champion ranking: profit vs targeting depth (seed-42 test split)")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, "day4_profit_curve")
plt.show()

print(f"break-even depth : {curve.break_even_depth}")
print(f"recommended depth: {curve.recommended_depth}  ->  ${curve.recommended_profit_usd:,.2f}")
check = [p for p in curve.points if p.depth == 0.3]
if check:
    p30 = check[0]
    print(f"\nat the ledger's 30%: ${p30.profit_usd:,.2f}  "
          f"({p30.incremental_conversions} incr. conv ± {p30.incremental_conversions_se}, "
          f"CI [{p30.profit_ci_low_usd:,.0f}, {p30.profit_ci_high_usd:,.0f}])")
    print(f"registry says      : ${meta['metrics']['profit_at_k_usd']:,.2f}  <- same construction")

**Reading.** The curve's point estimate breaks even at 2% and peaks at the
ledger's 30% — where it reproduces the registry's own metric to the cent
(same construction, two entry points). The honest part is the band: the 95%
interval clears zero only at some depths (4-8%, 14%, 24-32%) and includes it
at 16 of 25 — the wobble between neighbouring depths (8% dipping at 10%) is
single-split noise, not a cliff. One split with 81 converters can say "no
depth is obviously underwater"; it cannot certify an optimal depth. That
question is for an experiment sized by section 8, not for this curve.

## 4. Scenario comparison: the champion's money vs honest alternatives

Same budget, same holdout, four strategies: the champion's ranking, random
targeting (the null — what unmodelled money looks like), targeting everyone,
and doing nothing. `target_all` is flagged infeasible whenever the budget
cannot cover contacting the whole list.

In [ ]:
comp = compare_scenarios(y_money_te, score_champion, t_te, assumptions=ASSUMPTIONS, depth=0.3)
scen = pd.DataFrame([s.model_dump() for s in comp.scenarios])
print(scen.to_string(index=False))
print(f"\nbudget: ${comp.budget_usd:,.2f}   champion wins on mean: {comp.champion_wins}")

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = [s.strategy for s in comp.scenarios]
profits = [s.profit_usd for s in comp.scenarios]
err_lo = [s.profit_usd - s.profit_ci_low_usd for s in comp.scenarios]
err_hi = [s.profit_ci_high_usd - s.profit_usd for s in comp.scenarios]
colors = ["C2" if s.feasible else "0.8" for s in comp.scenarios]
ax.bar(labels, profits, color=colors, edgecolor="black", lw=0.5)
ax.errorbar(labels, profits, yerr=[err_lo, err_hi], fmt="none", ecolor="black", capsize=4)
ax.axhline(0, color="black", lw=0.8)
for i, s in enumerate(comp.scenarios):
    tag = "" if s.feasible else "  (infeasible @ budget)"
    ax.text(i, s.profit_usd, f" ${s.profit_usd:,.0f}{tag}", va="bottom", ha="center", fontsize=9)
ax.set_ylabel("profit on holdout (USD)")
ax.set_title("Same budget, four strategies (30% depth, seed-42 test split)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_fig(fig, "day4_scenario_comparison")
plt.show()

**Reading.** Three honest statements, no spin: (1) the champion leads the
random null on the mean ($858 vs $435) but the intervals overlap — the Day-3
tie, restated in dollars; (2) *target-all is not a losing strategy* — every
contacted converter pays $34.91 against a $0.10 contact, so blanketing the
list earns $1,451 in absolute terms — but it needs 3.3x the budget at hand
and earns $0.17/contact against the champion's $0.34, so at a fixed 30%
budget it is correctly marked infeasible; (3) doing nothing earns nothing.
The tool's job is to make the budget constraint visible, not to declare a
knockout.

## 5. Multi-treatment: "which email, if any?"

Hillstrom randomised three arms (Mens / Womens / No E-Mail). A binary
champion can only answer "email or not"; the money question is *which* email.
We fit one DR-learner per email arm against control, rank everyone by each
arm's predicted visit-lift, and define the policy: **send the argmax-lift arm
if its lift beats zero, otherwise nothing** — optionally restricted to the
top 30% of the lift list to match a budget.

Scoring such a policy honestly needs **inverse-propensity weighting**: a
customer randomised to Womens tells you nothing directly about Mens, so we
value the policy on the customers whose randomized arm happens to match what
the policy would send, upweighted by the arm count (propensity 1/3). The
policy is fit on train and valued on the untouched test rows, on the
**money outcome** (conversion) with visit as the ranking signal.

In [ ]:
df_multi = fetch_hillstrom(arm="multi")
ARMS = {"control": "No E-Mail", "mens": "Mens E-Mail", "womens": "Womens E-Mail"}
Xm = df_multi[FEATURES]
ym = df_multi["visit"]
tm = df_multi["segment"].astype(str)

splits_m = make_splits(Xm, ym, tm, random_state=RANDOM_STATE)
enc_m, _ = encode_splits(splits_m)
Xm_tr, ym_tr, tm_tr = enc_m["train"]
Xm_te, ym_te, tm_te = enc_m["test"]
ym_money_te = df_multi.loc[Xm_te.index, "conversion"].to_numpy(dtype=float)

# One DR-learner per email arm vs control, fit on train; score the test rows.
lifts = {}
for arm in ("mens", "womens"):
    fit = tm_tr.isin([ARMS[arm], ARMS["control"]]).to_numpy()
    model = make_learner("dr", target="visit", random_state=RANDOM_STATE)
    model.fit(Xm_tr.to_numpy()[fit], (tm_tr[fit] == ARMS[arm]).astype(int).to_numpy(),
              ym_tr.to_numpy()[fit])
    lifts[arm] = np.asarray(model.predict(Xm_te.to_numpy())).ravel()

arm_code = tm_te.map({"No E-Mail": 0, "Mens E-Mail": 1, "Womens E-Mail": 2}).to_numpy()
lift_matrix = np.vstack([np.zeros(len(lifts["mens"])), lifts["mens"], lifts["womens"]])

policy_argmax = lift_matrix.argmax(axis=0)
policy_argmax[lift_matrix.max(axis=0) <= 0] = 0  # no email beats doing nothing
top30 = lift_matrix.max(axis=0) >= np.quantile(lift_matrix.max(axis=0), 0.7)
policy_argmax30 = np.where(top30, policy_argmax, 0)

policies = {
    "no email (baseline)": np.zeros(len(arm_code), dtype=int),
    "mens for all": np.ones(len(arm_code), dtype=int),
    "womens for all": np.full(len(arm_code), 2),
    "argmax email (all)": policy_argmax,
    "argmax email (top 30%)": policy_argmax30,
}
vals = {
    name: policy_value_ipw(arm_code, ym_money_te, pol, assumptions=ASSUMPTIONS, strategy=name)
    for name, pol in policies.items()
}
rows = pd.DataFrame([v.model_dump() for v in vals.values()])
print(rows[["strategy", "n_customers", "incremental_conversions",
            "incremental_ci_low", "incremental_ci_high", "profit_usd"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
rows = pd.DataFrame([v.model_dump() for v in vals.values()])
ypos = np.arange(len(rows))
mean = rows["incremental_conversions"].to_numpy()
# The schema reports the CI on the per-customer rate; scale it to counts.
lo_conv = rows["incremental_ci_low"].to_numpy() * rows["n_customers"].to_numpy()
hi_conv = rows["incremental_ci_high"].to_numpy() * rows["n_customers"].to_numpy()
err = np.vstack([np.maximum(mean - lo_conv, 0), np.maximum(hi_conv - mean, 0)])
ax.barh(ypos, mean, xerr=err, color="C1", edgecolor="black", lw=0.5, capsize=4)
ax.set_yticks(ypos, rows["strategy"])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("incremental conversions on test (IPW, 95% CI)")
ax.set_title("Multi-arm policies, valued on conversions (seed-42 test split)")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
save_fig(fig, "day4_multi_treatment_policy")
plt.show()

for name, v in vals.items():
    contacted = int((policies[name] != 0).sum())
    print(f"{name:24s} emails {contacted:5d} ({contacted/len(arm_code):.0%})  "
          f"incr {v.incremental_conversions:7.1f}  profit ${v.profit_usd:,.0f}")

**Reading.** The argmax-email policy edges "mens for all" on the mean (51.0
vs 48.0 incremental conversions) — but the IPW intervals are wide enough
that this edge is noise (IPW pays for honesty with ~3x variance). Two
sharper results: the womens arm alone clears zero only barely (its CI
includes zero), and the *budget-limited* argmax (top 30%) lost money on this
split (-$70: nine incremental conversions cannot pay for 3,840 contacts).
What section 5 buys Day 6 is the capability — a typed, honest "which email,
if any?" answer with its uncertainty attached — not a precision claim. The
dollars are also not head-to-head with section 3: this test split comes from
the three-arm dataset (different rows).

## 6. Policy tree: segments a human can read

The five tools' segment profiler fits econml's honest `DRPolicyTree` on the
encoded training split (capped at depth 3 so every leaf is a sentence), then
**re-measures each segment on untouched holdout rows** — treated-minus-control
visit lift with a 95% interval, and the conversion lift alongside. The tree
proposes rules; the holdout grades them.

In [ ]:
profile = get_segment_profile(
    X_tr.to_numpy(), t_tr.to_numpy(), y_tr.to_numpy(),
    X_te.to_numpy(), t_te.to_numpy(), y_te.to_numpy(),
    feature_names=feature_names,
    y_eval_money=y_money_te,
    fairness_col=df.loc[X_te.index, "zip_code"].to_numpy(),
    max_depth=3,
    min_samples_leaf=500,
    random_state=RANDOM_STATE,
)

seg = pd.DataFrame([s.model_dump() for s in profile.segments])
cols = ["segment_id", "n_eval", "share", "policy_arm", "uplift_visit",
        "uplift_visit_ci_low", "uplift_visit_ci_high", "uplift_conversion"]
print(seg[cols].to_string(index=False))
print()
for s in profile.segments:
    print(f"segment {s.segment_id}: {', '.join(s.rules) if s.rules else '(root, no rule)'}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ypos = np.arange(len(seg))
mean = seg["uplift_visit"].to_numpy()
err = np.vstack([mean - seg["uplift_visit_ci_low"].to_numpy(),
                 seg["uplift_visit_ci_high"].to_numpy() - mean])
colors = ["C2" if p == "email" else "0.75" for p in seg["policy_arm"]]
ax.barh(ypos, mean, xerr=err, color=colors, edgecolor="black", lw=0.5, capsize=4)
ax.set_yticks(ypos, [f"seg {i} (n={n}, {sh:.0%})" for i, n, sh in
                     zip(seg["segment_id"], seg["n_eval"], seg["share"], strict=True)])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("holdout visit uplift (treated − control, 95% CI)")
ax.set_title("Policy-tree segments, re-measured on the holdout")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
save_fig(fig, "day4_policy_segments")
plt.show()

**Reading.** Two things a marketer can argue with: the rules themselves (a
split on `zip_code_Urban` or `history > 468` either matches domain knowledge
or provokes a data question — both good), and the holdout grading, which
**contradicts** the tree's own labels here: every segment's uplift interval
excludes zero, including the three the tree marked "no email". At these
economics (even the weakest segment's 0.053 lift times $34.91 is ~$1.85,
far above the $0.10 contact) the profit-maximising action is to email every
segment — the tree's capped-depth labels rank relative effect, they are not a
profit verdict. Read the tree as an effect map; read section 4 for the money.

## 7. Fairness audit: who does the policy touch?

Same policy, sliced by `zip_code`: the share of each group the policy would
actually email, and that group's own holdout uplift with its interval. Two
failure modes hide here — *disparate contact* (a group targeted far above its
population share) and *unsupported contact* (targeting a group whose uplift
interval includes zero).

In [ ]:
fair = pd.DataFrame([f.model_dump() for f in profile.fairness])
pop_share = df.loc[X_te.index, "zip_code"].value_counts(normalize=True)
fair["population_share"] = fair["group"].map(pop_share).round(4)
print(fair.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
ypos = np.arange(len(fair))
ax1.barh(ypos, fair["targeting_rate"], color="C0", edgecolor="black", lw=0.5,
         label="policy contact rate")
ax1.scatter(fair["population_share"], ypos, color="crimson", zorder=3, label="population share")
ax1.set_yticks(ypos, fair["group"])
ax1.set_xlabel("share")
ax1.set_title("Contact rate vs population share")
ax1.legend(fontsize=8)
ax1.grid(axis="x", alpha=0.3)

mean = fair["uplift_visit"].to_numpy()
err = np.vstack([mean - fair["uplift_visit_ci_low"].to_numpy(),
                 fair["uplift_visit_ci_high"].to_numpy() - mean])
ax2.barh(ypos, mean, xerr=err, color="C1", edgecolor="black", lw=0.5, capsize=4)
ax2.set_yticks(ypos, fair["group"])
ax2.axvline(0, color="black", lw=0.8)
ax2.set_xlabel("holdout visit uplift (95% CI)")
ax2.set_title("Per-group uplift")
ax2.grid(axis="x", alpha=0.3)
fig.tight_layout()
save_fig(fig, "day4_fairness_zip")
plt.show()

**Reading.** The audit found a real *contact* disparity: Urban customers
would be emailed at 59% against a 41% population share (the policy tree
splits on `zip_code_Urban` — a proxy-attribute smell), while Rural and
Suburban sit near their shares. The *effect* side is clean: all three groups'
uplift intervals exclude zero and overlap each other, so no group-level
effect disparity is supported — the overall claim carries, group-level ones
do not. The contact disparity is exactly what Day 6's adversarial eval must
catch: a copilot recommending urban-heavy targeting owes the user the
zip-proxy caveat, and `get_model_card`'s fairness note says it out loud.

## 8. Power analysis: how big must the next test be?

Everything above widens or narrows with sample size. Hillstrom's control arm
visits at ~10.6%; the question a marketer asks is "a +X-point lift — how many
customers, how long?". `design_ab_test` answers with the two-proportion sample
size and a duration against a daily eligible pool.

In [ ]:
baseline = float(df.loc[t == 0, "visit"].mean())
print(f"measured control visit rate: {baseline:.4f}")

mdes = np.linspace(0.005, 0.03, 25)
designs = [design_ab_test(baseline, float(m), daily_eligible=2000) for m in mdes]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot([d.mde_absolute for d in designs], [d.n_per_arm for d in designs], color="C0", lw=2)
chosen = design_ab_test(baseline, 0.01, daily_eligible=2000)
ax.scatter([chosen.mde_absolute], [chosen.n_per_arm], color="crimson", zorder=3)
ax.annotate(f"+1pp: {chosen.n_per_arm:,}/arm\n({chosen.duration_days} days @ 2,000/day)",
            (chosen.mde_absolute, chosen.n_per_arm), textcoords="offset points",
            xytext=(12, -6), fontsize=9)
ax.set_yscale("log")
ax.set_xlabel("MDE (absolute points on the visit rate)")
ax.set_ylabel("customers per arm (log scale)")
ax.set_title(f"Sample size to detect a lift (baseline {baseline:.1%}, 80% power, 5% alpha)")
ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, "day4_power_curve")
plt.show()

for m in (0.005, 0.01, 0.02, 0.03):
    d = design_ab_test(baseline, m, daily_eligible=2000)
    print(f"MDE +{m*100:.1f}pp (+{d.mde_relative:.0%}): {d.n_per_arm:,}/arm, "
          f"{d.n_total:,} total, {d.duration_days} days @ 2,000/day")

**Reading.** Detecting +1 point on a 10.6% visit rate needs roughly the whole
Hillstrom-scale list — which is exactly why Day 3's single-split intervals
were wide, and why the benchmark leaned on *repetition* (10 seeds) instead of
pretending one split could certify a 0.005 Qini gap. The design number that
matters for the next test is chosen by the *business's* minimum interesting
lift, not by what the current data can barely see; `design_ab_test` turns
that choice into a staffing decision (days at the daily eligible pool).

## 9. The five tools, end to end

Day 4's done-criterion: all five tools callable from Python with typed
schemas. Each section above used one; this cell is the smoke test Day 6's
FastAPI wrappers will mirror one-to-one.

In [ ]:
card = get_model_card("champion")
print("get_model_card      ->", card.name, card.learner, card.target,
      "| qini", card.metrics["qini_auc"], "| breaks:", len(card.when_not_to_use), "usage rules")

small = optimize_budget(y_money_te[:2000], score_champion[:2000], t_te[:2000],
                        assumptions=ASSUMPTIONS, n_depths=8)
print("optimize_budget     ->", len(small.points), "points,",
      "break-even", small.break_even_depth, "max@", small.recommended_depth)

comp_small = compare_scenarios(y_money_te, score_champion, t_te, depth=0.1)
print("compare_scenarios   ->", len(comp_small.scenarios), "strategies,",
      "champion wins:", comp_small.champion_wins)

print("get_segment_profile ->", len(profile.segments), "segments,",
      len(profile.fairness), "fairness rows")

d = design_ab_test(baseline, 0.015, daily_eligible=2000)
print("design_ab_test      ->", f"{d.n_per_arm:,}/arm,", d.duration_days, "days")

## 10. What this buys Day 5

- a **registered, provenance-checked champion** the copilot can cite by name
  (numbers from the card, never re-derived);
- a **profit curve and scenario table** as typed objects — the "recommendation,
  targets, intervals, baseline, assumptions, risks" the Day-5 brief template
  needs;
- **segment rules in sentences** for the brief's target list, with the
  holdout-graded caveat attached;
- an honest **"which email"** capability and a **power-aware** framing for the
  next experiment.

Caveat ledger (all measured, all carried forward): one-split profit intervals
cross zero at every depth; the champion-vs-random gap is inside its own noise;
IPW multi-arm intervals are wide by construction; no group-level causal claim
survives the fairness slice. Day 5's copilot must *quote* these limits, not
smooth them over.